# PREPROCESSING

### 0. Import and Load Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df_train = pd.read_csv("/Users/annya/DAC/Dataset/RawDataset/train.csv")
df_test = pd.read_csv("/Users/annya/DAC/Dataset/RawDataset/test.csv")

### 1. Drop Column Variance = 0

In [3]:
cols_to_drop = ['dx2_u00_u99', 'dx2_koo_k93', 'procv00_v89']

df_train = df_train.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

### 2. Transform Coloumn Count Sparse to become Biner Indikator

In [4]:
sparse_cols = ['proc28_28', 'proc76_77', 'proc29_31', 'proc24_27', 'proc46_51',
               'proc58_62', 'proc63_67', 'dx2_q00_q99', 'dx2_f00_f99', 'dx2_l00_l99',
               'dx2_v01_y98', 'proc78_79', 'dx2_h60_h95', 'proc_32_38', 'dx2_s00_t98',
               'proc68_70', 'dx2_c00_d48', 'proce00_e99']

for col in sparse_cols:
    df_train[col] = (df_train[col] > 0).astype(int)
    df_test[col] = (df_test[col] > 0).astype(int)

### 3. Seperate claim_id and label from feature

In [5]:
train_ids = df_train['claim_id']
test_ids = df_test['claim_id']

df_train = df_train.drop(columns=['claim_id'])
df_test = df_test.drop(columns=['claim_id'])

In [6]:
X = df_train.drop(columns=['label'])
y = df_train['label']

### 4. Train and Val Split (80:20)

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

### 5. One-Hot Encoding

In [8]:
onehot_cols = ['jkpst', 'jnspelsep', 'severitylevel']

X_train = pd.get_dummies(X_train, columns=onehot_cols)
X_val = pd.get_dummies(X_val, columns=onehot_cols)
df_test_encoded = pd.get_dummies(df_test, columns=onehot_cols)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
df_test_encoded = df_test_encoded.reindex(columns=X_train.columns, fill_value=0)

### 5. Target Encoding

In [9]:
high_card_cols = ['kdkc', 'dati2', 'typeppk', 'cmg', 'diagprimer']
smoothing = 10 
global_mean = y_train.mean()

encoding_maps = {}

for col in high_card_cols:
    stats = y_train.groupby(X_train[col]).agg(['mean', 'count'])
    smoothed = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
    encoding_maps[col] = smoothed

    X_train[col] = X_train[col].map(smoothed).fillna(global_mean)
    X_val[col] = X_val[col].map(smoothed).fillna(global_mean)
    df_test_encoded[col] = df_test_encoded[col].map(smoothed).fillna(global_mean)

(for tree based model)

In [10]:
X_train['los'] = np.log1p(X_train['los'])
X_val['los'] = np.log1p(X_val['los'])
df_test_encoded['los'] = np.log1p(df_test_encoded['los'])

### 6. Save Preprocessing

In [14]:
X_train.to_csv("/Users/annya/DAC/Dataset/Processed/X_train.csv", index=False)
X_val.to_csv("/Users/annya/DAC/Dataset/Processed/X_val.csv", index=False)
df_test_encoded.to_csv("/Users/annya/DAC/Dataset/Processed/X_test.csv", index=False)

y_train.to_csv("/Users/annya/DAC/Dataset/Processed/y_train.csv", index=False)
y_val.to_csv("/Users/annya/DAC/Dataset/Processed/y_val.csv", index=False)

train_ids.to_csv("/Users/annya/DAC/Dataset/Processed/train_ids.csv", index=False)
test_ids.to_csv("/Users/annya/DAC/Dataset/Processed/test_ids.csv", index=False)